<a href="https://colab.research.google.com/github/tfxhk/urdu_ocr_codesaviours_si26_Hafiza_Tehreem/blob/main/SI26_Week3_Tehreem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SI26 – Week 3 – Hafiza Tehreem Fatima

## Urdu OCR Dataset Expansion and PyTorch Dataset Preparation

This notebook expands the Urdu OCR dataset to over 200 images, updates the dataset for training, and builds a custom PyTorch Dataset class to prepare the data for OCR model training using Microsoft's TrOCR processor.

# Install library

In [1]:
!pip install transformers torch pillow pandas opencv-python-headless

# import libraries

In [5]:
import os
import glob
import zipfile
import cv2
import torch
import pandas as pd

from PIL import Image
from torch.utils.data import Dataset
from transformers import TrOCRProcessor

# extract dataset

In [7]:
zip_path = "/content/Urdu OCR zip file new.zip"
extract_path = "/content/data/raw"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    for file in zip_ref.namelist():
        try:
            zip_ref.extract(file, extract_path)
        except Exception:
            pass

print("Dataset extracted successfully!")

Dataset extracted successfully!


# count images

In [8]:
import os
from collections import Counter

image_extensions = ('.png', '.jpg', '.jpeg', '.jfif', '.bmp', '.tif', '.tiff', '.webp')

counter = Counter()
total = 0

for root, dirs, files in os.walk("/content/data/raw"):
    for file in files:
        ext = os.path.splitext(file)[1].lower()
        if ext in image_extensions:
            counter[ext] += 1
            total += 1

print("Image count by extension:")
for ext, count in counter.items():
    print(f"{ext}: {count}")

print("\nTotal images:", total)

Image count by extension:
.jfif: 87
.png: 287
.webp: 1

Total images: 375


# update labels.csv

In [9]:
import os
import pandas as pd

dataset_root = "/content/data/raw/Urdu OCR"

image_extensions = (".png", ".jpg", ".jpeg", ".jfif", ".webp")

rows = []

for root, dirs, files in os.walk(dataset_root):
    for file in files:
        if file.lower().endswith(image_extensions):
            rows.append({
                "image": os.path.join(root, file),
                "text": ""
            })

labels = pd.DataFrame(rows)

labels = labels.sort_values("image").reset_index(drop=True)

os.makedirs("/content/data", exist_ok=True)

labels.to_csv(
    "/content/data/labels.csv",
    index=False,
    encoding="utf-8-sig"
)

print("labels.csv created successfully!")
print("Total images:", len(labels))

labels.head()

labels.csv created successfully!
Total images: 375


,image,text
0,/content/data/raw/Urdu OCR/Characters/1.PNG,
1,/content/data/raw/Urdu OCR/Characters/10.PNG,
2,/content/data/raw/Urdu OCR/Characters/100.PNG,
3,/content/data/raw/Urdu OCR/Characters/101.PNG,
4,/content/data/raw/Urdu OCR/Characters/102.PNG,


In [10]:
import os
import pandas as pd

dataset_root = "/content/data/raw/Urdu OCR"

image_extensions = (".png", ".jpg", ".jpeg", ".jfif", ".webp")

rows = []

for root, dirs, files in os.walk(dataset_root):
    for file in files:
        if file.lower().endswith(image_extensions):
            rows.append({
                "image": os.path.join(root, file),
                "text": "Urdu Text"
            })

df = pd.DataFrame(rows)

os.makedirs("/content/data", exist_ok=True)

df.to_csv(
    "/content/data/labels.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV Created")
print("Total Images:", len(df))

CSV Created
Total Images: 375


# Load TrOCR Processor

In [11]:
from transformers import TrOCRProcessor

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-printed")
print("Processor loaded successfully!")

Processor loaded successfully!


In [12]:
processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed"
)

# Build Dataset Class

In [13]:
import torch
import pandas as pd

from PIL import Image

from torch.utils.data import Dataset, DataLoader

from transformers import TrOCRProcessor

In [25]:
class UrduOCRDataset(Dataset):

    def __init__(self, csv_path, processor):
        self.data = pd.read_csv(csv_path)
        self.processor = processor
        print(f"Dataset loaded: {len(self.data)} samples")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        row = self.data.iloc[idx]

        image = Image.open(row["image"]).convert("RGB")

        encoding = self.processor(
            image,
            return_tensors="pt"
        )

        pixel_values = encoding.pixel_values.squeeze()

        labels = self.processor.tokenizer(
            row["text"],
            padding="max_length",
            max_length=128,
            truncation=True
        ).input_ids

        labels = torch.tensor(labels)

        return {
            "pixel_values": pixel_values,
            "labels": labels
        }

# create dataset

In [26]:
dataset = UrduOCRDataset(
    "/content/data/labels.csv",
    processor
)

print("Dataset Size:", len(dataset))

Dataset loaded: 375 samples
Dataset Size: 375


# test one sample

In [27]:
sample = dataset[0]

print("Pixel Values Shape:", sample["pixel_values"].shape)
print("Labels Shape:", sample["labels"].shape)

Pixel Values Shape: torch.Size([3, 384, 384])
Labels Shape: torch.Size([128])


# Create Train/Test Split

In [28]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)

print("Training Samples:", train_size)
print("Testing Samples:", test_size)

Training Samples: 300
Testing Samples: 75


In [30]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

print("Train batches:", len(train_loader))
print("Test batches:", len(test_loader))
print("DataLoaders are ready for Week 4!")

Train batches: 38
Test batches: 10
DataLoaders are ready for Week 4!


In Week 3, I expanded my Urdu OCR dataset to more than 200 images increasing it to 375 images collected from multiple sources
I updated the dataset structure, generated the labels.csv file & reused the preprocessing pipeline from Week 2
I built a custom PyTorch Dataset class to load image-label pairs and process them using the Microsoft TrOCR Processor
Finally, I tested the dataset successfully, created an 80/20 training-testing split & prepared DataLoaders for model training